In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from inversion.pez import inversion_attack
from inversion.general import set_seed
import sys
import os
sys.path.append(os.path.abspath('..'))


# 1. Load Model and Tokenizer
model_id = 'google/gemma-2-2b-it'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map=device,
    trust_remote_code=True
)

for parameter in model.parameters():
    parameter.requires_grad = False

tokenizer.pad_token = tokenizer.eos_token

# 2. Define a sample prompt
prompt = "What are you doing?"
print(f"Original prompt: {prompt}")

# 3. Set inversion parameters
layer_idx = 2 
optimizer_cls = torch.optim.Adam
lr = 0.1
seed = 42

# 4. Run the inversion attack
match, inversion_time, iters, token_ids = inversion_attack(
    prompt,
    model,
    tokenizer,
    layer_idx,
    optimizer_cls,
    lr,
    seed=seed,
    scheduler=True,
    baseline=False
)

# 5. Print results
if match:
    print(f"Successfully inverted the prompt in {inversion_time:.2f} seconds with {iters} iterations.")
else:
    print("Failed to invert the prompt.")


